In [3]:
import pandas as pd

metadata = pd.read_csv("/content/HAM10000_metadata")

print("Number of images:", len(metadata))

print("\nColumns:")
print(metadata.columns.tolist())

print("\nClass distribution:")
print(metadata["dx"].value_counts())

print("\nFirst 5 rows:")
display(metadata.head())

Number of images: 10015

Columns:
['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']

Class distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

First 5 rows:


,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern


In [4]:
# Keep only the three MediShield classes
target_classes = ["bkl", "mel", "nv"]

target_data = metadata[metadata["dx"].isin(target_classes)].copy()

print("Images in our 3 target classes:")
print(target_data["dx"].value_counts())

print("\nUnique lesions in each class:")
print(target_data.groupby("dx")["lesion_id"].nunique())

print("\nImages per lesion:")
lesion_counts = (
    target_data
    .groupby(["dx", "lesion_id"])
    .size()
    .reset_index(name="image_count")
)

print(lesion_counts.groupby("dx")["image_count"].value_counts().sort_index())

Images in our 3 target classes:
dx
nv     6705
mel    1113
bkl    1099
Name: count, dtype: int64

Unique lesions in each class:
dx
bkl     727
mel     614
nv     5403
Name: lesion_id, dtype: int64

Images per lesion:
dx   image_count
bkl  1               440
     2               220
     3                55
     4                 8
     5                 2
     6                 2
mel  1               230
     2               278
     3               100
     4                 4
     5                 1
     6                 1
nv   1              4415
     2               699
     3               268
     4                18
     5                 2
     6                 1
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

# Reproducibility
RANDOM_STATE = 42

# Work only with our three target classes
data = metadata[metadata["dx"].isin(["bkl", "mel", "nv"])].copy()

# Get one row per lesion
lesions = data[["lesion_id", "dx"]].drop_duplicates()

print("Total unique target lesions:", len(lesions))
print("\nLesions per class:")
print(lesions["dx"].value_counts())

# Split lesions into train and temporary set
train_lesions, temp_lesions = train_test_split(
    lesions,
    test_size=0.30,
    stratify=lesions["dx"],
    random_state=RANDOM_STATE
)

# Split temporary set into validation and test
val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    stratify=temp_lesions["dx"],
    random_state=RANDOM_STATE
)

print("\nLesion-level split:")
print("Train lesions:", len(train_lesions))
print("Validation lesions:", len(val_lesions))
print("Test lesions:", len(test_lesions))

print("\nTrain:")
print(train_lesions["dx"].value_counts())

print("\nValidation:")
print(val_lesions["dx"].value_counts())

print("\nTest:")
print(test_lesions["dx"].value_counts())

Total unique target lesions: 6744

Lesions per class:
dx
nv     5403
bkl     727
mel     614
Name: count, dtype: int64

Lesion-level split:
Train lesions: 4720
Validation lesions: 1012
Test lesions: 1012

Train:
dx
nv     3781
bkl     509
mel     430
Name: count, dtype: int64

Validation:
dx
nv     811
bkl    109
mel     92
Name: count, dtype: int64

Test:
dx
nv     811
bkl    109
mel     92
Name: count, dtype: int64


In [6]:
# Convert lesion IDs into sets for easy membership checking
train_lesion_ids = set(train_lesions["lesion_id"])
val_lesion_ids = set(val_lesions["lesion_id"])
test_lesion_ids = set(test_lesions["lesion_id"])

# Assign every image to its lesion-level split
data["split"] = data["lesion_id"].apply(
    lambda x: (
        "train" if x in train_lesion_ids
        else "val" if x in val_lesion_ids
        else "test"
    )
)

print("IMAGE COUNTS BY SPLIT")
print(data.groupby(["split", "dx"]).size())

print("\nTOTAL IMAGES PER SPLIT")
print(data["split"].value_counts())

print("\nCHECK: lesion overlap")
print("Train ∩ Val:", len(train_lesion_ids & val_lesion_ids))
print("Train ∩ Test:", len(train_lesion_ids & test_lesion_ids))
print("Val ∩ Test:", len(val_lesion_ids & test_lesion_ids))

IMAGE COUNTS BY SPLIT
split  dx 
test   bkl     170
       mel     167
       nv      992
train  bkl     763
       mel     784
       nv     4698
val    bkl     166
       mel     162
       nv     1015
dtype: int64

TOTAL IMAGES PER SPLIT
split
train    6245
val      1343
test     1329
Name: count, dtype: int64

CHECK: lesion overlap
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [7]:
# Save the lesion-level split information

split_metadata = data[
    ["lesion_id", "image_id", "dx", "split"]
].copy()

split_path = "/content/MediShield/split_metadata.csv"

split_metadata.to_csv(split_path, index=False)

print("Saved:", split_path)
print("Rows:", len(split_metadata))

print("\nSplit counts:")
print(split_metadata["split"].value_counts())

OSError: Cannot save file into a non-existent directory: '/content/MediShield'

In [8]:
import os

os.makedirs("/content/MediShield", exist_ok=True)

split_metadata = data[
    ["lesion_id", "image_id", "dx", "split"]
].copy()

split_path = "/content/MediShield/split_metadata.csv"

split_metadata.to_csv(split_path, index=False)

print("Saved:", split_path)
print("Rows:", len(split_metadata))

print("\nSplit counts:")
print(split_metadata["split"].value_counts())

Saved: /content/MediShield/split_metadata.csv
Rows: 8917

Split counts:
split
train    6245
val      1343
test     1329
Name: count, dtype: int64


In [9]:
for split in ["train", "val", "test"]:
    print(f"\n===== {split.upper()} =====")

    for cls in ["bkl", "mel", "nv"]:
        subset = data[
            (data["split"] == split) &
            (data["dx"] == cls)
        ]

        print(
            f"{cls}: {len(subset)} images | "
            f"{subset['lesion_id'].nunique()} lesions"
        )


===== TRAIN =====
bkl: 763 images | 509 lesions
mel: 784 images | 430 lesions
nv: 4698 images | 3781 lesions

===== VAL =====
bkl: 166 images | 109 lesions
mel: 162 images | 92 lesions
nv: 1015 images | 811 lesions

===== TEST =====
bkl: 170 images | 109 lesions
mel: 167 images | 92 lesions
nv: 992 images | 811 lesions


In [10]:
# Create a balanced training subset
# We keep all bkl images and select the same number of mel/nv images.

RANDOM_STATE = 42

train_data = data[data["split"] == "train"].copy()

# Number of images to use from each class
train_target = train_data[train_data["dx"] == "bkl"].shape[0]

print("Training target per class:", train_target)

balanced_train_parts = []

for cls in ["bkl", "mel", "nv"]:
    class_data = train_data[train_data["dx"] == cls]

    selected = class_data.sample(
        n=train_target,
        random_state=RANDOM_STATE
    )

    balanced_train_parts.append(selected)

balanced_train = pd.concat(
    balanced_train_parts,
    ignore_index=True
)

print("\nBalanced training set:")
print(balanced_train["dx"].value_counts())

print("\nTotal training images:", len(balanced_train))

Training target per class: 763

Balanced training set:
dx
bkl    763
mel    763
nv     763
Name: count, dtype: int64

Total training images: 2289


In [11]:
# Create balanced validation and test subsets

val_data = data[data["split"] == "val"].copy()
test_data = data[data["split"] == "test"].copy()

def make_balanced_subset(df, target_per_class, seed=42):
    parts = []

    for cls in ["bkl", "mel", "nv"]:
        class_data = df[df["dx"] == cls]

        selected = class_data.sample(
            n=target_per_class,
            random_state=seed
        )

        parts.append(selected)

    return pd.concat(parts, ignore_index=True)


# Validation: 162 images per class
balanced_val = make_balanced_subset(
    val_data,
    target_per_class=162
)

# Test: 167 images per class
balanced_test = make_balanced_subset(
    test_data,
    target_per_class=167
)

print("VALIDATION")
print(balanced_val["dx"].value_counts())
print("Total:", len(balanced_val))

print("\nTEST")
print(balanced_test["dx"].value_counts())
print("Total:", len(balanced_test))


VALIDATION
dx
bkl    162
mel    162
nv     162
Name: count, dtype: int64
Total: 486

TEST
dx
bkl    167
mel    167
nv     167
Name: count, dtype: int64
Total: 501


In [12]:
# Save the final selected datasets

final_train = balanced_train[["lesion_id", "image_id", "dx"]].copy()
final_val = balanced_val[["lesion_id", "image_id", "dx"]].copy()
final_test = balanced_test[["lesion_id", "image_id", "dx"]].copy()

final_train.to_csv("/content/MediShield/train.csv", index=False)
final_val.to_csv("/content/MediShield/val.csv", index=False)
final_test.to_csv("/content/MediShield/test.csv", index=False)

# Also create one combined file
all_selected = pd.concat(
    [
        final_train.assign(split="train"),
        final_val.assign(split="val"),
        final_test.assign(split="test")
    ],
    ignore_index=True
)

all_selected.to_csv(
    "/content/MediShield/selected_images.csv",
    index=False
)

print("Saved successfully!")
print()
print("Train:", len(final_train))
print("Validation:", len(final_val))
print("Test:", len(final_test))
print("Total:", len(all_selected))

print("\nFiles:")
print("/content/MediShield/train.csv")
print("/content/MediShield/val.csv")
print("/content/MediShield/test.csv")
print("/content/MediShield/selected_images.csv")

Saved successfully!

Train: 2289
Validation: 486
Test: 501
Total: 3276

Files:
/content/MediShield/train.csv
/content/MediShield/val.csv
/content/MediShield/test.csv
/content/MediShield/selected_images.csv


In [13]:
from google.colab import files

files.download("/content/MediShield/selected_images.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import zipfile
import os

zip_path = "/content/MediShield_selected_images.zip"

print("ZIP exists:", os.path.exists(zip_path))

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    jpg_files = [f for f in files if f.lower().endswith(".jpg")]

print("Total files in ZIP:", len(files))
print("JPEG images:", len(jpg_files))

print("\nFirst 5 files:")
for f in jpg_files[:5]:
    print(f)

ZIP exists: True


BadZipFile: File is not a zip file

In [15]:
import os

zip_path = "/content/MediShield_selected_images.zip"

size_bytes = os.path.getsize(zip_path)
size_gb = size_bytes / (1024**3)

print("File size:", size_bytes, "bytes")
print("File size:", round(size_gb, 3), "GB")

File size: 5242880 bytes
File size: 0.005 GB


In [16]:
import os

zip_path = "/content/MediShield_selected_images.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

print("Incomplete ZIP removed.")

Incomplete ZIP removed.


In [17]:
import os
import glob

parts = sorted(glob.glob("/content/MediShield_part_*.bin"))

print("Parts found:", len(parts))

total_size = 0

for part in parts:
    size = os.path.getsize(part)
    total_size += size
    print(os.path.basename(part), "→", size, "bytes")

print("\nTotal uploaded size:", total_size, "bytes")
print("Expected size:      928160558 bytes")
print("Size matches:", total_size == 928160558)

Parts found: 5
MediShield_part_00.bin → 1048576 bytes
MediShield_part_01.bin → 1048576 bytes
MediShield_part_02.bin → 1048576 bytes
MediShield_part_03.bin → 1048576 bytes
MediShield_part_04.bin → 1048576 bytes

Total uploaded size: 5242880 bytes
Expected size:      928160558 bytes
Size matches: False


In [18]:
import os

print("Free space:", round(os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / (1024**3), 2), "GB")

Free space: 65.22 GB


In [19]:
!pip -q install datasets

In [20]:
from datasets import load_dataset

dataset = load_dataset("kuchikihater/HAM10000")

print(dataset)

README.md:   0%|          | 0.00/647 [00:00<?, ?B/s]

data/train-00000-of-00006-6e9cc1f2a2bc29(…): reconstructing file:   0%|          |  0.00B /  481MB            

data/train-00000-of-00006-6e9cc1f2a2bc29(…): downloading bytes:           |  0.00B            

data/train-00001-of-00006-d0e18093f7a3c6(…): reconstructing file:   0%|          |  0.00B /  478MB            

data/train-00001-of-00006-d0e18093f7a3c6(…): downloading bytes:           |  0.00B            

...

data/train-00004-of-00006-7a1958d1ffb42c(…): downloading bytes:           |  0.00B            

data/train-00005-of-00006-bcb0987169f383(…): reconstructing file:   0%|          |  0.00B /  407MB            

data/train-00005-of-00006-bcb0987169f383(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10015 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 10015
    })
})


In [21]:
print(dataset["train"].features)

print("\nFirst 10 labels:")
print(dataset["train"]["label"][:10])

print("\nLabel names:")
print(dataset["train"].features["label"].names)

{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'])}

First 10 labels:
[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Label names:
['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [22]:
from collections import Counter

labels = dataset["train"]["label"]

counts = Counter(labels)

print("Class distribution:")
for label_id, count in sorted(counts.items()):
    print(f"{dataset['train'].features['label'].names[label_id]:6s} : {count}")

Class distribution:
akiec  : 327
bcc    : 514
bkl    : 1099
df     : 115
mel    : 1113
nv     : 6705
vasc   : 142


In [23]:
import random
from collections import defaultdict

# Reproducibility
SEED = 42
random.seed(SEED)

# Original label IDs
label_names = dataset["train"].features["label"].names

# Our three classes
target_classes = ["bkl", "mel", "nv"]
target_ids = [label_names.index(c) for c in target_classes]

print("Target classes:", target_classes)
print("Target IDs:", target_ids)

# Collect indices for each class
class_indices = defaultdict(list)

for i, label in enumerate(dataset["train"]["label"]):
    if label in target_ids:
        class_indices[label].append(i)

# Select 500 per class
selected_indices = {}

for class_name, class_id in zip(target_classes, target_ids):
    indices = class_indices[class_id].copy()
    random.shuffle(indices)
    selected_indices[class_name] = indices[:500]

    print(f"{class_name}: {len(selected_indices[class_name])} images")

print("\nTotal selected:",
      sum(len(v) for v in selected_indices.values()))

Target classes: ['bkl', 'mel', 'nv']
Target IDs: [2, 4, 5]
bkl: 500 images
mel: 500 images
nv: 500 images

Total selected: 1500


In [25]:
from sklearn.model_selection import train_test_split

train_indices = []
val_indices = []
test_indices = []

for class_name in target_classes:
    indices = selected_indices[class_name]

    # 70% train, 30% temporary
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.30,
        random_state=SEED
    )

    # Split remaining 30% equally → 15% val, 15% test
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=SEED
    )

    train_indices.extend(train_idx)
    val_indices.extend(val_idx)
    test_indices.extend(test_idx)

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))
print("Total:", len(train_indices) + len(val_indices) + len(test_indices))

Train: 1050
Validation: 225
Test: 225
Total: 1500


In [28]:
print(train_data.columns.tolist())

['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset', 'split']


In [29]:
# ============================================================
# MediShield - PyTorch Dataset & DataLoaders
# ============================================================

import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Class mapping
# ------------------------------------------------------------

class_to_idx = {
    "bkl": 0,
    "mel": 1,
    "nv": 2
}

idx_to_class = {
    0: "bkl",
    1: "mel",
    2: "nv"
}

# ------------------------------------------------------------
# Image directory
# ------------------------------------------------------------

IMAGE_DIR = "/content/MediShield/data/images"

print("Image directory:", IMAGE_DIR)
print("Directory exists:", os.path.exists(IMAGE_DIR))

# ------------------------------------------------------------
# Image transformations
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# PyTorch Dataset
# ------------------------------------------------------------

class HAMDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        # Image filename
        image_id = row["image_id"]
        image_path = os.path.join(
            self.image_dir,
            image_id + ".jpg"
        )

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Convert class name to numerical label
        diagnosis = row["dx"]
        label = class_to_idx[diagnosis]

        # Apply transformation
        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = HAMDataset(
    train_data,
    IMAGE_DIR,
    transform=train_transform
)

val_dataset = HAMDataset(
    val_data,
    IMAGE_DIR,
    transform=eval_transform
)

test_dataset = HAMDataset(
    test_data,
    IMAGE_DIR,
    transform=eval_transform
)

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

print()
print("=" * 60)
print("MEDISHIELD PYTORCH DATALOADER CHECK")
print("=" * 60)

print(f"Train samples      : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)}")

print()

print(f"Batch image shape  : {images.shape}")
print(f"Batch label shape  : {labels.shape}")

print()

print(f"Labels in batch    : {sorted(labels.unique().tolist())}")

print()

print("Class mapping:")
print("0 -> bkl")
print("1 -> mel")
print("2 -> nv")

print()

print(f"Image dtype        : {images.dtype}")
print(f"Label dtype        : {labels.dtype}")

print()

print("DataLoader setup successful!")
print("=" * 60)

Image directory: /content/MediShield/data/images
Directory exists: False


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1818/2663990043.py", line 100, in __getitem__
    image = Image.open(image_path).convert("RGB")
            ~~~~~~~~~~^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No such file or directory: '/content/MediShield/data/images/ISIC_0027331.jpg'


In [30]:
# Check what type of data we currently have

print("train_data type:", type(train_data))
print("Number of train rows:", len(train_data))

print("\nFirst row:")
print(train_data.iloc[0])

train_data type: <class 'pandas.core.frame.DataFrame'>
Number of train rows: 6245

First row:
lesion_id        HAM_0000118
image_id        ISIC_0027419
dx                       bkl
dx_type                histo
age                     80.0
sex                     male
localization           scalp
dataset         vidir_modern
split                  train
Name: 0, dtype: object


In [31]:
# ============================================================
# MediShield - Save Selected Images Locally
# ============================================================

import os

IMAGE_DIR = "/content/MediShield/data/images"
os.makedirs(IMAGE_DIR, exist_ok=True)

print("Saving selected images...")
print("Target directory:", IMAGE_DIR)

# The original Hugging Face dataset
# contains all 10,015 HAM10000 images.
#
# We use the image_id from our selected metadata
# to locate the corresponding image.

# Create a lookup from image_id -> image
hf_dataset = dataset["train"]

image_lookup = {}

for i in range(len(hf_dataset)):
    image_id = hf_dataset[i]["image_id"] if "image_id" in hf_dataset.column_names else None

    if image_id is not None:
        image_lookup[image_id] = hf_dataset[i]["image"]

print("Image lookup created:", len(image_lookup))

# ------------------------------------------------------------
# Collect image IDs from our selected datasets
# ------------------------------------------------------------

selected_df = pd.concat(
    [train_data, val_data, test_data],
    ignore_index=True
)

selected_df = selected_df.drop_duplicates(
    subset=["image_id"]
)

print("Selected unique images:", len(selected_df))

# ------------------------------------------------------------
# Save images
# ------------------------------------------------------------

saved = 0
missing = []

for _, row in selected_df.iterrows():

    image_id = row["image_id"]
    output_path = os.path.join(
        IMAGE_DIR,
        image_id + ".jpg"
    )

    if os.path.exists(output_path):
        saved += 1
        continue

    if image_id not in image_lookup:
        missing.append(image_id)
        continue

    image = image_lookup[image_id].convert("RGB")
    image.save(output_path, "JPEG", quality=95)

    saved += 1

print()
print("=" * 60)
print("IMAGE EXTRACTION COMPLETE")
print("=" * 60)
print("Images saved :", saved)
print("Images missing:", len(missing))
print("Directory    :", IMAGE_DIR)
print("=" * 60)

Saving selected images...
Target directory: /content/MediShield/data/images
Image lookup created: 0
Selected unique images: 8917

IMAGE EXTRACTION COMPLETE
Images saved : 0
Images missing: 8917
Directory    : /content/MediShield/data/images


In [32]:
print(type(dataset))
print(dataset)

<class 'datasets.dataset_dict.DatasetDict'>
DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 10015
    })
})


In [33]:
# ============================================================
# MediShield - Final Dataset + PyTorch DataLoaders
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SEED = 42
torch.manual_seed(SEED)

TARGET_PER_CLASS = 500

# HF labels:
# 2 = bkl
# 4 = mel
# 5 = nv

original_to_class = {
    2: 0,
    4: 1,
    5: 2
}

idx_to_class = {
    0: "bkl",
    1: "mel",
    2: "nv"
}

hf_dataset = dataset["train"]

print("Total HAM10000 images:", len(hf_dataset))

# ------------------------------------------------------------
# Select 500 images from each target class
# ------------------------------------------------------------

selected_indices = []

for original_label in [2, 4, 5]:

    class_indices = [
        i for i, label in enumerate(hf_dataset["label"])
        if label == original_label
    ]

    # Reproducible selection
    generator = torch.Generator().manual_seed(SEED + original_label)

    shuffled = torch.randperm(
        len(class_indices),
        generator=generator
    ).tolist()

    chosen = [
        class_indices[i]
        for i in shuffled[:TARGET_PER_CLASS]
    ]

    selected_indices.extend(chosen)

    print(
        idx_to_class[original_to_class[original_label]],
        "selected:",
        len(chosen)
    )

print()
print("Total selected:", len(selected_indices))

# ------------------------------------------------------------
# Shuffle selected data
# ------------------------------------------------------------

generator = torch.Generator().manual_seed(SEED)

shuffle_order = torch.randperm(
    len(selected_indices),
    generator=generator
).tolist()

selected_indices = [
    selected_indices[i]
    for i in shuffle_order
]

# ------------------------------------------------------------
# Split 70 / 15 / 15
# ------------------------------------------------------------

labels_for_split = [
    original_to_class[hf_dataset["label"][i]]
    for i in selected_indices
]

train_indices, temp_indices = train_test_split(
    selected_indices,
    test_size=0.30,
    stratify=labels_for_split,
    random_state=SEED
)

temp_labels = [
    original_to_class[hf_dataset["label"][i]]
    for i in temp_indices
]

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)

print()
print("Split sizes:")
print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

# ------------------------------------------------------------
# Image transformations
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# PyTorch Dataset
# ------------------------------------------------------------

class HAMHFDataset(Dataset):

    def __init__(self, hf_dataset, indices, transform=None):

        self.dataset = hf_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        real_index = self.indices[idx]

        item = self.dataset[real_index]

        image = item["image"].convert("RGB")

        original_label = item["label"]

        label = original_to_class[original_label]

        if self.transform is not None:
            image = self.transform(image)

        return image, label


# ------------------------------------------------------------
# Create PyTorch datasets
# ------------------------------------------------------------

train_dataset = HAMHFDataset(
    hf_dataset,
    train_indices,
    train_transform
)

val_dataset = HAMHFDataset(
    hf_dataset,
    val_indices,
    eval_transform
)

test_dataset = HAMHFDataset(
    hf_dataset,
    test_indices,
    eval_transform
)

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

images, labels = next(iter(train_loader))

print()
print("=" * 60)
print("MEDISHIELD DATASET CHECK")
print("=" * 60)

print("Train samples      :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Test samples       :", len(test_dataset))

print()

print("Batch image shape  :", images.shape)
print("Batch label shape  :", labels.shape)

print()

print("Labels in batch    :", sorted(labels.unique().tolist()))

print()

print("Class mapping:")
print("0 -> bkl")
print("1 -> mel")
print("2 -> nv")

print()

print("Image dtype        :", images.dtype)
print("Label dtype        :", labels.dtype)

print()

print("DataLoader setup successful!")
print("=" * 60)

Total HAM10000 images: 10015
bkl selected: 500
mel selected: 500
nv selected: 500

Total selected: 1500

Split sizes:
Train: 1050
Validation: 225
Test: 225

MEDISHIELD DATASET CHECK
Train samples      : 1050
Validation samples : 225
Test samples       : 225

Batch image shape  : torch.Size([32, 3, 224, 224])
Batch label shape  : torch.Size([32])

Labels in batch    : [0, 1, 2]

Class mapping:
0 -> bkl
1 -> mel
2 -> nv

Image dtype        : torch.float32
Label dtype        : torch.int64

DataLoader setup successful!
